# Modulo 5 · Lezione 6 — Documentazione e best practice

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | OpenAPI e Swagger — documentazione automatica e arricchita |
| 2 | CORS — permettere chiamate dai frontend |
| 3 | Middleware — logging, tempo di risposta |
| 4 | Rate limiting — protezione dagli abusi |
| 5 | Versionamento delle API |
| 6 | Best practice di design REST |

In [ ]:
import subprocess, sys

for pkg in ['fastapi', 'uvicorn[standard]', 'requests', 'slowapi']:
    try:
        __import__(pkg.split('[')[0])
        print(f"{pkg.split('[')[0]}: già installato")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
        print(f"{pkg}: installato")

---
## 1. OpenAPI e Swagger

FastAPI genera automaticamente la specifica OpenAPI 3.0. Puoi arricchirla con metadati:

In [ ]:
from fastapi import FastAPI, Query, Path, Body
from fastapi.openapi.utils import get_openapi
from pydantic import BaseModel, Field
from typing import Optional
import uvicorn, threading, time, requests

# Metadati dell'app
app = FastAPI(
    title="API di Proyectos v1",
    description="""
## API per la gestione di progetti di team

Permette di creare e gestire progetti, task e membri del team.

### Autenticazione
Usa Bearer token JWT. Ottieni il token su `/token`.

### Limiti
- 100 richieste/minuto per IP
- Payload massimo: 1MB
""",
    version="1.0.0",
    contact={"name": "Team API", "email": "api@empresa.com"},
    license_info={"name": "MIT"},
    openapi_tags=[
        {"name": "Proyectos", "description": "Gestione dei progetti"},
        {"name": "Tareas",    "description": "Task all'interno dei progetti"},
        {"name": "Salud",     "description": "Stato del sistema"},
    ]
)


class ProyectoCreate(BaseModel):
    nombre:      str           = Field(
        min_length=1, max_length=100,
        description="Nome univoco del progetto",
        examples=["Web App v2", "API Migration"]
    )
    descripcion: Optional[str] = Field(None, description="Descrizione dettagliata")
    presupuesto: Optional[float] = Field(None, gt=0, description="In euro")

    model_config = {
        "json_schema_extra": {
            "examples": [{
                "nombre": "Rediseño portal",
                "descripcion": "Nuova versione del portale clienti",
                "presupuesto": 15000.0
            }]
        }
    }

class ProyectoResponse(ProyectoCreate):
    id: int


proyectos_db: dict[int, dict] = {}
seq = 0

@app.get("/salud", tags=["Salud"], summary="Stato del servizio")
def healthcheck():
    """Restituisce lo stato attuale del servizio e la sua versione."""
    return {"status": "ok", "version": "1.0.0"}


@app.get(
    "/proyectos",
    tags=["Proyectos"],
    summary="Elenca i progetti",
    description="Restituisce tutti i progetti con supporto alla paginazione.",
    response_model=list[ProyectoResponse],
    responses={
        200: {"description": "Elenco dei progetti"},
        401: {"description": "Non autenticato"},
    }
)
def listar(
    skip:  int = Query(0,  ge=0,   description="Elementi da saltare"),
    limit: int = Query(20, ge=1, le=100, description="Numero massimo di risultati")
):
    items = list(proyectos_db.values())
    return items[skip:skip+limit]


@app.post(
    "/proyectos",
    tags=["Proyectos"],
    summary="Crea un progetto",
    response_model=ProyectoResponse,
    status_code=201,
    responses={
        201: {"description": "Progetto creato"},
        409: {"description": "Nome duplicato"},
        422: {"description": "Dati non validi"},
    }
)
def crear(datos: ProyectoCreate):
    global seq
    if any(p['nombre'] == datos.nombre for p in proyectos_db.values()):
        from fastapi import HTTPException
        raise HTTPException(409, "Esiste già un progetto con questo nome")
    seq += 1
    p = {"id": seq, **datos.model_dump()}
    proyectos_db[seq] = p
    return p


cfg = uvicorn.Config(app, host="127.0.0.1", port=8030, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)
B = "http://127.0.0.1:8030"

print("Spec OpenAPI disponibile su:")
print(f"  {B}/openapi.json")
print(f"  {B}/docs        (Swagger UI)")
print(f"  {B}/redoc       (ReDoc)")

spec = requests.get(f"{B}/openapi.json").json()
print(f"\nVersione spec: {spec['openapi']}")
print(f"Titolo:        {spec['info']['title']}")
print(f"Endpoint:      {list(spec['paths'].keys())}")

---
## 2. CORS

```
CORS (Cross-Origin Resource Sharing):

Senza CORS:
  Frontend su https://miapp.com → API su https://api.miapp.com → ❌ BLOCCATO

Con CORS configurato:
  Frontend su https://miapp.com → API → ✓ header CORS → ✓ OK

Il browser invia:
  OPTIONS /endpoint  Origin: https://miapp.com

L'API risponde:
  Access-Control-Allow-Origin: https://miapp.com
  Access-Control-Allow-Methods: GET, POST, PUT, DELETE
  Access-Control-Allow-Headers: Authorization, Content-Type
```

In [ ]:
from fastapi.middleware.cors import CORSMiddleware

app_cors = FastAPI(title="API con CORS")

# CORS — controlla quali origini possono chiamare l'API
app_cors.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:3000",       # frontend dev
        "https://miapp.com",           # produzione
        "https://admin.miapp.com",     # pannello admin
    ],
    allow_credentials=True,            # permette cookie e header di auth
    allow_methods=["GET", "POST", "PUT", "PATCH", "DELETE", "OPTIONS"],
    allow_headers=["Authorization", "Content-Type", "X-Request-ID"],
    expose_headers=["X-Total-Count"],  # header che il JS può leggere
    max_age=600,                       # tempo di cache del preflight (secondi)
)

@app_cors.get("/datos")
def datos():
    return {"mensaje": "Con CORS configurato"}

print("CORS configurato:")
print("  - http://localhost:3000 (dev frontend)")
print("  - https://miapp.com (produzione)")
print("  - https://admin.miapp.com (admin)")
print("\nPer API pubbliche usa: allow_origins=['*']")
print("⚠️  '*' non funziona con allow_credentials=True")

---
## 3. Middleware

In [ ]:
from fastapi import FastAPI, Request, Response
from starlette.middleware.base import BaseHTTPMiddleware
import time, uuid, logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("api")

app_mid = FastAPI(title="API con Middleware")


# ── Middleware 1: Logging delle request ────────────────────────
class LoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        request_id = str(uuid.uuid4())[:8]
        inicio      = time.time()

        # Prima dell'endpoint
        logger.info(f"[{request_id}] → {request.method} {request.url.path}")

        response = await call_next(request)

        # Dopo l'endpoint
        duracion = (time.time() - inicio) * 1000
        logger.info(f"[{request_id}] ← {response.status_code} ({duracion:.1f}ms)")

        # Aggiungere header diagnostici
        response.headers["X-Request-ID"]      = request_id
        response.headers["X-Response-Time"]   = f"{duracion:.1f}ms"
        return response


# ── Middleware 2: Sicurezza (security headers) ────────────────
class SecurityHeadersMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        response = await call_next(request)
        response.headers["X-Content-Type-Options"]   = "nosniff"
        response.headers["X-Frame-Options"]           = "DENY"
        response.headers["X-XSS-Protection"]          = "1; mode=block"
        response.headers["Strict-Transport-Security"] = "max-age=31536000"
        return response


app_mid.add_middleware(LoggingMiddleware)
app_mid.add_middleware(SecurityHeadersMiddleware)

@app_mid.get("/rapido")
def rapido(): return {"msg": "risposta immediata"}

@app_mid.get("/lento")
async def lento():
    import asyncio
    await asyncio.sleep(0.2)  # simulare un'operazione lenta
    return {"msg": "risposta dopo 200ms"}


cfg = uvicorn.Config(app_mid, host="127.0.0.1", port=8031, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)
B_MID = "http://127.0.0.1:8031"

r = requests.get(f"{B_MID}/rapido")
print("Header diagnostici:")
for h in ["X-Request-ID", "X-Response-Time", "X-Content-Type-Options"]:
    print(f"  {h}: {r.headers.get(h, 'non presente')}")

r2 = requests.get(f"{B_MID}/lento")
print(f"\nEndpoint lento: {r2.headers.get('X-Response-Time')}")

---
## 4. Rate Limiting

In [ ]:
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded
from fastapi import FastAPI, Request

limiter = Limiter(key_func=get_remote_address)  # identifica per IP
app_rl  = FastAPI(title="API con Rate Limiting")
app_rl.state.limiter = limiter
app_rl.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)

@app_rl.get("/datos")
@limiter.limit("5/minute")       # 5 richieste al minuto per IP
async def datos(request: Request):
    return {"datos": [1, 2, 3]}

@app_rl.get("/pesado")
@limiter.limit("2/minute")       # endpoint costosi → limite più stretto
async def pesado(request: Request):
    return {"resultado": "operazione costosa"}

@app_rl.get("/publico")
async def publico():              # nessun limite
    return {"info": "accesso libero"}


cfg = uvicorn.Config(app_rl, host="127.0.0.1", port=8032, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)
B_RL = "http://127.0.0.1:8032"

print("Test del rate limit (5/min):")
for i in range(7):
    r = requests.get(f"{B_RL}/datos")
    estado = "✓" if r.status_code == 200 else f"✗ {r.status_code}"
    print(f"  Richiesta {i+1}: {estado}")

---
## 5. Versionamento delle API

```
Strategie di versionamento:

1. URL path (la più comune):
   GET /v1/usuarios
   GET /v2/usuarios

2. Header:
   Accept: application/vnd.miapi.v2+json

3. Query param:
   GET /usuarios?version=2

Quando versionare:
✓ Cambio che rompe la compatibilità (breaking change)
✓ Cambio nella struttura della risposta
✓ Rimozione di campi

Non è necessario versionare:
- Aggiungere campi opzionali (non rompe i client esistenti)
- Nuovi endpoint
- Correzione di bug
```

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

# ── Router per versione ──────────────────────────────────────
from fastapi import APIRouter

v1 = APIRouter(prefix="/v1", tags=["v1"])
v2 = APIRouter(prefix="/v2", tags=["v2"])

# V1: risposta semplice
class UsuarioV1(BaseModel):
    id:     int
    nombre: str
    email:  str

# V2: aggiunge avatar, metadata, cambia 'nombre' → 'nombre_completo'
class UsuarioV2(BaseModel):
    id:             int
    nombre_completo:str
    email:          str
    avatar_url:     str | None = None
    metadata:       dict       = {}

USUARIOS = [
    {"id": 1, "nombre": "Ana García",   "email": "ana@x.com"},
    {"id": 2, "nombre": "Carlos López", "email": "carlos@x.com"},
]

@v1.get("/usuarios", response_model=list[UsuarioV1])
def usuarios_v1():
    return USUARIOS

@v2.get("/usuarios", response_model=list[UsuarioV2])
def usuarios_v2():
    # Trasformare nel nuovo formato
    return [
        {"id": u["id"], "nombre_completo": u["nombre"], "email": u["email"],
         "avatar_url": None, "metadata": {"migrado_de": "v1"}}
        for u in USUARIOS
    ]

@v1.get("/usuarios/{id}", response_model=UsuarioV1)
def usuario_v1(id: int):
    u = next((u for u in USUARIOS if u["id"] == id), None)
    if not u: from fastapi import HTTPException; raise HTTPException(404, "Non trovato")
    return u

@v2.get("/usuarios/{id}", response_model=UsuarioV2)
def usuario_v2(id: int):
    u = next((u for u in USUARIOS if u["id"] == id), None)
    if not u: from fastapi import HTTPException; raise HTTPException(404, "Non trovato")
    return {"id": u["id"], "nombre_completo": u["nombre"], "email": u["email"],
            "avatar_url": f"https://avatars.example.com/{u['id']}.png", "metadata": {}}


app_ver = FastAPI(title="API versionata")
app_ver.include_router(v1)
app_ver.include_router(v2)

# Header di deprecation su v1
@app_ver.middleware("http")
async def deprecation_header(request: Request, call_next):
    response = await call_next(request)
    if request.url.path.startswith("/v1"):
        response.headers["Deprecation"]  = "true"
        response.headers["Sunset"]       = "2026-12-31T00:00:00Z"
        response.headers["Link"]         = '</v2/usuarios>; rel="successor-version"'
    return response


cfg = uvicorn.Config(app_ver, host="127.0.0.1", port=8033, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)
B_VER = "http://127.0.0.1:8033"

print("V1 (deprecated):")
r1 = requests.get(f"{B_VER}/v1/usuarios")
print(f"  {r1.json()}")
print(f"  Deprecation: {r1.headers.get('Deprecation', 'no')}")
print(f"  Sunset:      {r1.headers.get('Sunset', 'no')}")

print("\nV2 (attuale):")
r2 = requests.get(f"{B_VER}/v2/usuarios")
print(f"  {r2.json()}")

---
## 6. Best practice di design REST

```
✓ NOMI DELLE RISORSE
  ✓  /usuarios          (plurale, sostantivo)
  ✓  /usuarios/{id}/pedidos
  ✗  /getUsuarios       (verbi, no)
  ✗  /usuario/{id}/getPedidos

✓ VERBI HTTP ↔ OPERAZIONI
  GET    /proyectos       → elencare
  POST   /proyectos       → creare
  GET    /proyectos/42    → ottenerne uno
  PUT    /proyectos/42    → sostituire completamente
  PATCH  /proyectos/42    → aggiornare parzialmente
  DELETE /proyectos/42    → eliminare

✓ STATUS CODE
  200 OK             → GET, PUT, PATCH riusciti
  201 Created        → POST riuscito
  204 No Content     → DELETE riuscito
  400 Bad Request    → dati malformati
  401 Unauthorized   → non autenticato
  403 Forbidden      → autenticato ma senza permessi
  404 Not Found      → risorsa inesistente
  409 Conflict       → conflitto di business (duplicato, stato errato)
  422 Unprocessable  → dati validi ma che violano regole (FastAPI lo usa per la validazione)
  429 Too Many Requests → rate limit
  500 Internal Error → errore del server

✓ FORMATO DEGLI ERRORI COERENTE
  {
    "error":   "NOMBRE_ERROR",
    "message": "Descrizione leggibile",
    "details": {...}
  }

✓ PAGINAZIONE
  GET /items?skip=0&limit=20
  Risposta: {"total": 100, "skip": 0, "limit": 20, "items": [...]}

✓ IDEMPOTENZA
  GET, PUT, DELETE → stessa chiamata = stesso risultato
  POST → può creare duplicati se chiamato più volte
  PATCH → generalmente idempotente se applica le stesse modifiche
```

In [ ]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError
from pydantic import BaseModel
from typing import Optional, Any

app_bp = FastAPI(title="API con buone pratiche")

# ── Formato di errore coerente ─────────────────────────────
class ErrorResponse(BaseModel):
    error:   str
    message: str
    details: Optional[Any] = None

# Intercettare gli errori di validazione Pydantic (422)
@app_bp.exception_handler(RequestValidationError)
async def validation_handler(request: Request, exc: RequestValidationError):
    errores = [
        {"campo": ".".join(str(x) for x in e["loc"][1:]), "mensaje": e["msg"]}
        for e in exc.errors()
    ]
    return JSONResponse(
        status_code=422,
        content={"error": "VALIDATION_ERROR", "message": "Dati non validi", "details": errores}
    )

# Intercettare HTTPException
@app_bp.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    return JSONResponse(
        status_code=exc.status_code,
        content={"error": f"HTTP_{exc.status_code}", "message": exc.detail, "details": None}
    )


from pydantic import Field

class ItemCreate(BaseModel):
    nombre: str = Field(min_length=1)
    precio: float = Field(gt=0)

items_db: dict[int, dict] = {}
seq2 = 0

@app_bp.get("/items")
def listar_items(skip: int = 0, limit: int = 20):
    todos  = list(items_db.values())
    return {"total": len(todos), "skip": skip, "limit": limit, "items": todos[skip:skip+limit]}

@app_bp.post("/items", status_code=201)
def crear_item(datos: ItemCreate):
    global seq2; seq2 += 1
    item = {"id": seq2, **datos.model_dump()}
    items_db[seq2] = item
    return item

@app_bp.get("/items/{id}")
def get_item(id: int):
    if id not in items_db:
        raise HTTPException(status_code=404, detail=f"Item {id} non trovato")
    return items_db[id]


cfg = uvicorn.Config(app_bp, host="127.0.0.1", port=8034, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)
B_BP = "http://127.0.0.1:8034"

print("Errore di validazione (422):")
r = requests.post(f"{B_BP}/items", json={"nombre": "", "precio": -5})
import json
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

print("\nNon trovato (404):")
r = requests.get(f"{B_BP}/items/999")
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

---
## Pratica

### Esercizio 1 — Documentazione arricchita

**Consegna:** Prendi una semplice API di gestione eventi (puoi crearla da zero o partire da quella dei task della Lezione 3) e aggiungi una documentazione completa:

- Metadati dell'app (titolo, descrizione con Markdown, versione, contatto)
- Tag per raggruppare gli endpoint
- `summary` e `description` per ogni endpoint
- `examples` nei modelli Pydantic
- `responses` che documentano i possibili status code di ogni endpoint

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field
from typing import Optional

app_ev = FastAPI(
    title="API Eventos",
    description="""
## Gestione di eventi e conferenze

Crea, consulta e gestisci eventi con controllo della capienza.

### Autenticazione
Bearer token JWT. Ottieni il token su `/token`.
""",
    version="1.0.0",
    contact={"name": "Supporto", "email": "soporte@eventos.com"},
    openapi_tags=[
        {"name": "Eventos",    "description": "CRUD degli eventi"},
        {"name": "Asistentes", "description": "Gestione delle iscrizioni"},
    ]
)

class EventoCreate(BaseModel):
    nombre:   str           = Field(min_length=1, max_length=200,
                                    examples=["PyDay Galicia 2025"])
    lugar:    str           = Field(min_length=1, examples=["Santiago de Compostela"])
    aforo:    int           = Field(gt=0, description="Posti massimi")
    fecha:    str           = Field(description="Formato YYYY-MM-DD", examples=["2025-10-15"])
    online:   bool          = False

    model_config = {
        "json_schema_extra": {
            "examples": [{
                "nombre": "PyDay Galicia 2025",
                "lugar":  "Auditorio USC",
                "aforo":  200,
                "fecha":  "2025-10-15",
                "online": False
            }]
        }
    }

class EventoResponse(EventoCreate):
    id:         int
    inscritos:  int = 0

ev_db: dict[int, dict] = {}
ev_seq = 0

@app_ev.get(
    "/eventos",
    tags=["Eventos"],
    summary="Elenca tutti gli eventi",
    description="Restituisce gli eventi disponibili con supporto a paginazione e filtri.",
    response_model=list[EventoResponse],
    responses={
        200: {"description": "Elenco degli eventi"},
    }
)
def listar_eventos(
    online: Optional[bool] = Query(None, description="Filtra per formato online/in presenza"),
    skip:   int            = 0,
    limit:  int            = Query(20, le=100)
):
    items = list(ev_db.values())
    if online is not None: items = [e for e in items if e['online'] == online]
    return items[skip:skip+limit]

@app_ev.post(
    "/eventos",
    tags=["Eventos"],
    summary="Crea evento",
    response_model=EventoResponse,
    status_code=201,
    responses={
        201: {"description": "Evento creato"},
        409: {"description": "Nome evento duplicato"},
        422: {"description": "Dati non validi"},
    }
)
def crear_evento(datos: EventoCreate):
    global ev_seq
    ev_seq += 1
    ev = {"id": ev_seq, "inscritos": 0, **datos.model_dump()}
    ev_db[ev_seq] = ev
    return ev
```
</details>

---
### Esercizio 2 — Middleware di metriche

**Consegna:** Implementa un middleware che registri le metriche d'uso:

- Contatore delle request per endpoint e metodo
- Tempo di risposta cumulativo per endpoint
- `GET /metricas` — restituisce le statistiche raccolte
- `POST /metricas/reset` — azzera i contatori

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
from fastapi import FastAPI, Request
from starlette.middleware.base import BaseHTTPMiddleware
import time
from collections import defaultdict

metricas: dict[str, dict] = defaultdict(lambda: {"requests": 0, "tiempo_total_ms": 0.0})

class MetricasMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        inicio   = time.time()
        response = await call_next(request)
        duracion = (time.time() - inicio) * 1000

        clave = f"{request.method} {request.url.path}"
        metricas[clave]["requests"]       += 1
        metricas[clave]["tiempo_total_ms"] += duracion

        return response

app_m = FastAPI()
app_m.add_middleware(MetricasMiddleware)

@app_m.get("/datos")  def datos(): return {"ok": True}
@app_m.get("/otros") def otros(): return {"ok": True}

@app_m.get("/metricas")
def ver_metricas():
    return {
        endpoint: {
            "requests": v["requests"],
            "tiempo_medio_ms": round(v["tiempo_total_ms"] / v["requests"], 1) if v["requests"] else 0
        }
        for endpoint, v in metricas.items()
        if not endpoint.endswith("/metricas")  # escludere l'endpoint stesso
    }

@app_m.post("/metricas/reset")
def reset_metricas():
    metricas.clear()
    return {"mensaje": "Metriche azzerate"}
```
</details>

---
### Esercizio 3 — CORS e header di sicurezza

**Consegna:** Configura CORS per due ambienti:

- In sviluppo (`ENV=dev`): permetti tutte le origini
- In produzione (`ENV=prod`): permetti solo `https://miapp.com` e `https://admin.miapp.com`
- Aggiungi un middleware che inietti header di sicurezza standard
- Endpoint `GET /config` che mostri l'ambiente attivo e le origini permesse

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import os
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from starlette.middleware.base import BaseHTTPMiddleware

ENV = os.getenv("ENV", "dev")  # dev di default

ORIGENES = {
    "dev":  ["*"],
    "prod": ["https://miapp.com", "https://admin.miapp.com"],
}

origenes_activos = ORIGENES.get(ENV, ["*"])

app_env = FastAPI()

app_env.add_middleware(
    CORSMiddleware,
    allow_origins=origenes_activos,
    allow_credentials=ENV == "prod",  # solo in prod con origini specifiche
    allow_methods=["*"],
    allow_headers=["Authorization", "Content-Type"],
)

SECURITY_HEADERS = {
    "X-Content-Type-Options":    "nosniff",
    "X-Frame-Options":           "DENY",
    "X-XSS-Protection":          "1; mode=block",
    "Referrer-Policy":           "strict-origin-when-cross-origin",
    "Permissions-Policy":        "camera=(), microphone=(), geolocation=()",
}

class SecurityHeaders(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        response = await call_next(request)
        for k, v in SECURITY_HEADERS.items():
            response.headers[k] = v
        if ENV == "prod":
            response.headers["Strict-Transport-Security"] = "max-age=31536000; includeSubDomains"
        return response

app_env.add_middleware(SecurityHeaders)

@app_env.get("/config")
def config():
    return {
        "entorno": ENV,
        "origenes_cors": origenes_activos,
        "headers_seguridad": list(SECURITY_HEADERS.keys()),
    }
```
</details>

---
### Esercizio 4 — Rate limiting per utente

**Consegna:** Implementa un rate limiting differenziato senza slowapi:

- Utenti autenticati: 100 richieste/minuto
- Utenti anonimi: 10 richieste/minuto
- Admin: nessun limite
- Implementalo come middleware che ispezioni l'header Authorization

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from starlette.middleware.base import BaseHTTPMiddleware
import time
from collections import defaultdict

# {clave: [(timestamp, )]}
ventana_requests: dict[str, list] = defaultdict(list)

LIMITES = {"anonimo": 10, "usuario": 100, "admin": None}  # None = nessun limite

def obtener_rol(token: str | None) -> str:
    if not token: return "anonimo"
    try:
        from jose import jwt
        payload = jwt.decode(token, CLAVE_SECRETA, algorithms=["HS256"])
        return payload.get("rol", "usuario")
    except: return "anonimo"

class RateLimitMiddleware(BaseHTTPMiddleware):
    VENTANA = 60  # secondi

    async def dispatch(self, request: Request, call_next):
        auth    = request.headers.get("Authorization", "")
        token   = auth.replace("Bearer ", "") if auth.startswith("Bearer ") else None
        rol     = obtener_rol(token)
        limite  = LIMITES.get(rol)

        if limite is not None:  # admin → nessun controllo
            ip    = request.client.host
            clave = f"{ip}:{rol}"
            ahora = time.time()

            # Rimuovere le request fuori dalla finestra
            ventana_requests[clave] = [
                t for t in ventana_requests[clave] if ahora - t < self.VENTANA
            ]

            if len(ventana_requests[clave]) >= limite:
                return JSONResponse(
                    status_code=429,
                    content={"error": "RATE_LIMIT", "message": f"Limite: {limite} req/min per {rol}"},
                    headers={"Retry-After": "60"}
                )
            ventana_requests[clave].append(ahora)

        return await call_next(request)
```
</details>

---
### Esercizio 5 (sfida) — API pronta per la produzione

**Consegna:** Combina tutto quanto appreso in questa lezione per realizzare un'API di livello produzione:

Prendi l'API dei task della Lezione 3 e aggiungi:
1. Documentazione OpenAPI completa (metadati, tag, examples, responses)
2. CORS configurato (dev vs. prod in base a una variabile d'ambiente)
3. Middleware di logging con `X-Request-ID` e tempo di risposta
4. Rate limiting: 20 req/min per IP
5. Gestione degli errori con formato coerente `{error, message, details}`
6. Versionamento: l'API attuale è `/v1/tareas`, aggiungi `/v2/tareas` dove la risposta include `tiempo_restante_dias` calcolato da `fecha_limite`

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

Questo esercizio è lungo — ecco la struttura chiave:

```python
# 1. App con metadati
app_prod = FastAPI(
    title="API Tareas v1",
    description="## API di gestione dei task\n\nSupporta CRUD completo con autenticazione.",
    version="1.0.0",
    openapi_tags=[{"name": "Tareas", "description": "Gestione dei task del team"}]
)

# 2. CORS
import os
app_prod.add_middleware(CORSMiddleware,
    allow_origins=["*"] if os.getenv("ENV","dev") == "dev" else ["https://miapp.com"],
    allow_methods=["*"], allow_headers=["*"]
)

# 3. Middleware di logging (vedi demo della lezione)
app_prod.add_middleware(LoggingMiddleware)

# 4. Rate limiting con slowapi (vedi demo)

# 5. Exception handler
@app_prod.exception_handler(RequestValidationError)
async def val_err(request, exc):
    return JSONResponse(422, {"error": "VALIDATION_ERROR", "message": "Dati non validi",
                               "details": exc.errors()})

# 6. Router v1 e v2
v1 = APIRouter(prefix="/v1", tags=["Tareas"])
v2 = APIRouter(prefix="/v2", tags=["Tareas v2"])

@v1.get("/tareas") def listar_v1(): return list(bd.values())

@v2.get("/tareas")
def listar_v2():
    from datetime import date
    resultado = []
    for t in bd.values():
        t2 = dict(t)
        if t.get("fecha_limite"):
            try:
                fl = date.fromisoformat(t["fecha_limite"])
                t2["tiempo_restante_dias"] = (fl - date.today()).days
            except: t2["tiempo_restante_dias"] = None
        resultado.append(t2)
    return resultado

app_prod.include_router(v1)
app_prod.include_router(v2)
```
</details>

---
## Riepilogo

```python
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from starlette.middleware.base import BaseHTTPMiddleware

app = FastAPI(
    title="La mia API",
    description="Descrizione con **Markdown**",
    version="1.0.0",
    openapi_tags=[{"name": "Recursos", "description": "..."}]
)

# CORS
app.add_middleware(CORSMiddleware,
    allow_origins=["https://miapp.com"],
    allow_methods=["*"], allow_headers=["Authorization", "Content-Type"]
)

# Middleware personalizzato
class MiMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        response = await call_next(request)
        response.headers["X-Custom"] = "valor"
        return response
app.add_middleware(MiMiddleware)

# Endpoint documentato
@app.get("/recursos", tags=["Recursos"], summary="Elenca le risorse",
         responses={200: {...}, 401: {...}})
def listar(): ...

# Versionamento con router
v1 = APIRouter(prefix="/v1")
v2 = APIRouter(prefix="/v2")
app.include_router(v1)
app.include_router(v2)
```

## Cosa vediamo nella prossima lezione?

Nella **Lezione 7** costruiamo il progetto finale del modulo: un'API completa con autenticazione, database, documentazione e tutte le best practice apprese.